# Aralin 17 - Paglikha ng Lokal na AI Agents gamit ang Foundry Local at Qwen

Sa notebook na ito ay gagawa ka ng isang **lokal na engineering assistant** na tumatakbo nang buo sa iyong workstation. Walang cloud inference na ginagamit kahit kailan. Ang assistant ay kayang:

1. **Tumawag ng mga tools** gamit ang Qwen function calling sa pamamagitan ng Foundry Local.
2. **Maglista at magbasa ng mga file** sa loob ng isang sandboxed na direktoryo ng proyekto.
3. **Suriin ang code** gamit ang simpleng mga sukatan.
4. **Maghanap ng dokumentasyon** gamit ang lokal na RAG (Chroma).
5. **Gumamit ng lokal na MCP server** (maingat na nilalaktawan kung wala).

Halos pareho ang hitsura ng agent code sa mga cloud lessons — ang nag-iiba lang ay ang client endpoint mula sa cloud papuntang `localhost`.


## Setup

Bago patakbuhin ang notebook na ito:

1. **I-install ang Microsoft Foundry Local** (tingnan ang [dokumentasyon](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) para sa iyong OS).
2. **I-download at simulan ang isang Qwen model:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. I-install ang mga package ng Python sa ibaba.

Lahat ay tumatakbo nang lokal. Isang makina na may ~8 GB RAM ang makatotohanang minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Kumonekta sa Foundry Local

I-dinadownload ng `FoundryLocalManager` ang modelo kung kinakailangan, sinisimulan ang lokal na serbisyo, at binibigyan tayo ng isang **endpoint na compatible sa OpenAI**. Pagkatapos ay itinuturo namin dito ang karaniwang OpenAI SDK. Ang API key ay isang lokal na placeholder — walang cloud credential na kasangkot.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Mga Lokal na Kasangkapan (Sandboxed File Operations)

Gumawa kami ng isang maliit na sample na proyekto sa disk, pagkatapos ay itinakda ang mga kasangkapan na sakop sa ugat ng proyektong iyon. Mahalaga ang sandbox check kahit lokal lang: ang isang kasangkapang nagbabasa ng mga arbitraryong path ay tumatakbo gamit ang mga permiso ng iyong user at maaaring hawakan ang anumang kaya mo.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokal na RAG gamit ang Chroma

Naglalagay kami ng maliit na set ng mga snippet ng dokumentasyon sa isang lokal na koleksyon ng Chroma. Tumakbo ang Chroma sa proseso at nag-iimbak ng mga vector sa disk — walang server, walang cloud. Kinukuha ng tool na `search_docs` ang pinaka-naugnay na mga snippet para sa isang query.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Ang Tool-Calling Loop

Ngayon ay nirerehistro natin ang mga tool sa model gamit ang OpenAI tools schema at pinapatakbo ang karaniwang tool-calling loop — humihiling ang modelo ng isang tool, pinapatakbo namin ito nang lokal, ibinabalik ang resulta, at inuulit hanggang makabuo ang modelo ng panghuling sagot. Ang maaasahang function calling ng Qwen ang dahilan kung bakit ito gumagana nang on-device.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokal na MCP (Opsyonal)

Ang MCP ay isang transport, hindi isang cloud service — ang MCP server ay maaaring tumakbo bilang lokal na proseso gamit ang `stdio`. Ipinapakita ng cell sa ibaba kung paano ikonekta ang isang lokal na MCP server kung mayroon kang isa na nakakonpigurang (halimbawa isang filesystem server). Ito ay maingat na nilalaktawan kapag hindi nakatakda ang `LOCAL_MCP_COMMAND`, kaya ang notebook ay tumatakbo pa rin nang buong proseso kahit wala ito.

Paalala sa seguridad: ang lokal na MCP server ay tumatakbo sa mga permiso ng iyong user. Limitahan ito sa direktoryo ng proyekto at suriin ang mga output bago kumilos batay dito.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Buod

Nakatayo ka ng isang engineering assistant na tumatakbo nang buo sa iyong makina:

- **Foundry Local** ang naglingkod ng **Qwen** model sa likod ng OpenAI-compatible endpoint — kaya ang agent code ay tugma sa mga aralin sa cloud.
- **Sandboxed tools** ang nagbigay sa agent ng access sa mga file at pagsusuri ng code nang hindi umaalis sa direktoryo ng proyekto.
- **Chroma** ang nagbigay ng **local RAG** sa dokumentasyon.
- **Local MCP** ang nagpakita kung paano muling gamitin ang MCP ecosystem offline.

Walang paggamit ng cloud inference sa kahit anong punto.

### Hamon

Palawakin ang local agent upang:

1. **Magtrabaho sa maraming MCP servers** — ikonekta ang isang filesystem server at isang git server at hayaang pumili ang agent sa pagitan nila.
2. **Gumamit ng lokal na memorya** — i-persist ang maikling kasaysayan ng pag-uusap sa disk upang tandaan ng assistant ang naunang mga turn kahit i-restart ang notebook.
3. **Suportahan ang lokal na multi-agent orchestration** — magdagdag ng pangalawang local agent (hal. isang reviewer) at hayaang mag-collaborate ang dalawa sa isang gawain.

Sa susunod na aralin matututuhan mo kung paano i-secure ang mga deployed AI agents.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
